목표:
- 평가 문서 30개 중 1개 문서를 대상으로 per-question 생성 답변을 평가
- 질문별로 gold/pred 및 지표(ret_recall, ret_mrr, gen_fill, gen_match, gen_sim)를 확인
- 문서 전체 평균도 함께 표시
- 결과는 json 1개로 저장 (doc index/id + 질문별 상세 + 문서 평균)

참고 사항:
- 기존 코드의 per-question 실험 구조와 추상화 클래스(RAGExperiment 등)를 그대로 사용
- retrieval 상세(ctx/chunk idx 등)는 저장/표시하지 않음
- gen_match/gen_sim은 evalgenpred()의 rapidfuzz token_set_ratio 기반(기존 구현 그대로)

In [1]:
import json
import re
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer

from preprocess.pp_basic import docs, BASE_DIR, GOLD_EVIDENCE_CSV, GOLD_FIELDS_JSONL
from preprocess.rag_experiment_per_question import (
    CONFIG,
    ExperimentSpec,
    load_questions_df,
    make_components,
    RAGExperiment,
)

d:\dev\github\codeit-part3-team4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv(), override=False)

client = OpenAI()
embedmodel = SentenceTransformer("nlpai-lab/KoE5")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 602.12it/s, Materializing param=pooler.dense.weight]                               


### Load gold (CSV + JSONL) + questions

In [3]:
gold_evidence_df = pd.read_csv(GOLD_EVIDENCE_CSV)

def load_gold_fields_jsonl(path: Path) -> pd.DataFrame:
    out = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)

            iid = r.get("instance_id")
            doc_id = r.get("doc_id")  # JSONL top-level key
            fields = r.get("fields", {}) or {}

            for k, v in fields.items():
                out.append({"instance_id": iid, "doc_id": doc_id, "field": k, "gold": v})

    return pd.DataFrame(out)

gold_fields_df = load_gold_fields_jsonl(Path(GOLD_FIELDS_JSONL))
questions_df = load_questions_df()

print("gold_evidence_df:", gold_evidence_df.shape, "cols:", gold_evidence_df.columns.tolist())
print("gold_fields_df:", gold_fields_df.shape, "cols:", gold_fields_df.columns.tolist())
print("gold_fields_df unique doc_id:", gold_fields_df["doc_id"].astype(str).nunique())
print("questions_df:", questions_df.shape, "cols:", questions_df.columns.tolist())
print("ndocs(all):", len(docs))

gold_evidence_df: (630, 5) cols: ['instance_id', 'doc_id', 'page_start', 'page_end', 'anchor_text']
gold_fields_df: (630, 4) cols: ['instance_id', 'doc_id', 'field', 'gold']
gold_fields_df unique doc_id: 30
questions_df: (311, 5) cols: ['instance_id', 'qid', 'doc_id', 'question', 'type']
ndocs(all): 100


### Build EVAL_DOCS + choose target by index

In [4]:
def namekey(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s)).strip()
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s+", " ", s)
    return s

gold_docid_set = set(namekey(x) for x in gold_fields_df["doc_id"].astype(str).dropna().unique())
EVAL_DOCS = [p for p in docs if namekey(p.name) in gold_docid_set]

print("Eval docs:", len(EVAL_DOCS))
print("Eval doc_id examples:", [p.name for p in EVAL_DOCS[:5]])

if not EVAL_DOCS:
    raise RuntimeError("EVAL_DOCS is empty. doc_id matching failed.")

# ====== 사용자 수정 포인트: 인덱스만 바꿔서 반복 실행 ======
TARGET_DOC_INDEX = 0
# =========================================================
target_doc_path = EVAL_DOCS[TARGET_DOC_INDEX]

print("TARGET_DOC_INDEX:", TARGET_DOC_INDEX)
print("TARGET_DOC_ID(doc_id):", target_doc_path.name)
print("TARGET_DOC_PATH:", str(target_doc_path))

def find_evaldoc_index_by_contains(substr: str) -> List[Tuple[int, str]]:
    hits = []
    for i, p in enumerate(EVAL_DOCS):
        if str(substr) in p.name:
            hits.append((i, p.name))
    return hits

print("Find 'D004' hits:", find_evaldoc_index_by_contains("D004")[:50])

Eval docs: 30
Eval doc_id examples: ['(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf', '(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.pdf', '(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.pdf', '(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.pdf', '2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.pdf']
TARGET_DOC_INDEX: 0
TARGET_DOC_ID(doc_id): (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf
TARGET_DOC_PATH: d:\dev\github\codeit-part3-team4\data\raw\files\(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf
Find 'D004' hits: []


### fix config + build components (spec=3 C1/R2/G1)

In [5]:
CONFIG["perquestionretrieve"] = True
CONFIG["perquestiongenerate"] = True
CONFIG["savecontextperquestion"] = False

spec = ExperimentSpec(exp_id=3, chunker="C1", retriever="R2", generator="G1")
print("Running spec:", spec)

chunker, retriever, generator = make_components(spec, embed_model=embedmodel, client=client)

rag = RAGExperiment(
    chunker=chunker,
    retriever=retriever,
    generator=generator,
    questions_df=questions_df,
)

Running spec: ExperimentSpec(exp_id=3, chunker='C1', retriever='R2', generator='G1')


### run single doc

In [6]:
SIM_THRESHOLD = 80
top_k = int(CONFIG.get("top_k", CONFIG.get("topk", 15)))

doc_metrics = rag.run_single_doc_metrics(
    doc_path=target_doc_path,
    gold_fields_df=gold_fields_df,
    gold_evidence_df=gold_evidence_df,
    top_k=top_k,
    sim_threshold=SIM_THRESHOLD,
    warn_on_mismatch=True,
)

print("doc_metrics keys:", list(doc_metrics.keys())[:50])
pd.DataFrame([doc_metrics]).head(1)

doc_metrics keys: ['doc_id', 'expected_answer_count', 'answer_count', 'n_questions', 'chunk_count', 'raw_text_len', 'raw_text_preview', 'answers_preview', 'n_nonempty_answers', 'n_notfound_answers', 'pred_preview', 'n_nonempty_preds', 'n_notfound_preds', 'pred_map', 'idxs_map', 'ctx_map', 'ret_recall', 'ret_mrr', 'gen_fill', 'gen_match', 'gen_sim']


,doc_id,expected_answer_count,answer_count,n_questions,chunk_count,raw_text_len,raw_text_preview,answers_preview,n_nonempty_answers,n_notfound_answers,...,n_nonempty_preds,n_notfound_preds,pred_map,idxs_map,ctx_map,ret_recall,ret_mrr,gen_fill,gen_match,gen_sim
0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .pdf,21,21,21,87,150,"{""performance_response_time"":""웹 페이지 조회 시 사용자가 ...","[벤처확인종합관리시스템 기능 고도화 용역사업, 벤처기업확인기관, 벤처기업육성법상 복...",21,1,...,21,1,"{'project_name': '벤처확인종합관리시스템 기능 고도화 용역사업', 'a...","{'project_name': [47, 48, 46, 67, 56, 73, 68, ...",None,0.952381,0.486732,1.0,0.238095,48.935502


### per-question detail (gold + pred + gen metrics)

In [7]:
# sentinel은 프로젝트 버전에 따라 다를 수 있어 "둘 다" 허용
SENT_NOTFOUND_SET = {"notfound", "not_found"}
SENT_GENFAIL_SET  = {"genfail", "gen_fail"}

def eval_gen_pred(pred: str, gold: Optional[str], threshold: int = 80) -> Dict[str, float]:
    pred = (pred or "").strip()
    predl = pred.lower()

    if predl in SENT_GENFAIL_SET or predl in SENT_NOTFOUND_SET:
        return {"gen_fill": 0.0, "gen_match": np.nan, "gen_sim": np.nan}
    if predl in {"", "none", "null"}:
        return {"gen_fill": 0.0, "gen_match": np.nan, "gen_sim": np.nan}

    if gold is None or str(gold).strip() == "":
        return {"gen_fill": 1.0, "gen_match": np.nan, "gen_sim": np.nan}

    golds = str(gold).strip()
    sim = float(fuzz.token_set_ratio(pred, golds))
    match = 1.0 if sim >= threshold else 0.0
    return {"gen_fill": 1.0, "gen_match": match, "gen_sim": sim}

COMMON_DOC_MARK = "*"  # questions_df에서 공통 질문 표시
Q_DOC_COL = "doc_id"   # 이미 확인됨: questions_df 컬럼에 doc_id 존재

def get_queries_for_doc(doc_id: str, questions_df: pd.DataFrame) -> List[Tuple[str, str]]:
    common = questions_df.loc[questions_df[Q_DOC_COL].astype(str) == COMMON_DOC_MARK, ["type", "question"]]
    perdoc = questions_df.loc[questions_df[Q_DOC_COL].astype(str) == doc_id, ["type", "question"]]

    merged = pd.concat([common, perdoc], ignore_index=True)
    merged["type"] = merged["type"].astype(str)
    merged["question"] = merged["question"].astype(str)

    # type 중복이면 per-doc가 common을 덮어쓰기(뒤에 있는 것을 keep)
    merged = merged.drop_duplicates(subset=["type"], keep="last")
    return list(zip(merged["type"].tolist(), merged["question"].tolist()))

doc_id = str(doc_metrics.get("doc_id", target_doc_path.name))

predmap = (
    doc_metrics.get("pred_map")
    or doc_metrics.get("predmap")
    or {}
)

queries = get_queries_for_doc(doc_id, questions_df)

gold_qdf = gold_fields_df.loc[gold_fields_df["doc_id"].astype(str) == doc_id, ["field", "gold"]].copy()
gold_qdf["field"] = gold_qdf["field"].astype(str)

rows = []
for t, q in queries:
    gold_row = gold_qdf.loc[gold_qdf["field"] == str(t)]
    gold = None if gold_row.empty else gold_row["gold"].iloc[0]

    pred = predmap.get(t, "NOTFOUND")  # 기본값은 관측 빈도 높은 쪽으로
    g = eval_gen_pred(pred=pred, gold=gold, threshold=SIM_THRESHOLD)

    rows.append(
        {
            "doc_index": int(TARGET_DOC_INDEX),
            "doc_id": doc_id,
            "type": str(t),
            "question": str(q),
            "gold": None if gold is None else str(gold),
            "pred": None if pred is None else str(pred),
            "gen_fill": float(g["gen_fill"]) if not pd.isna(g["gen_fill"]) else np.nan,
            "gen_match": float(g["gen_match"]) if not pd.isna(g["gen_match"]) else np.nan,
            "gen_sim": float(g["gen_sim"]) if not pd.isna(g["gen_sim"]) else np.nan,
        }
    )

detail_df = pd.DataFrame(rows)

detail_view = detail_df.sort_values(
    by=["gen_match", "gen_sim", "type"],
    ascending=[True, True, True],
    na_position="last",
).reset_index(drop=True)

detail_view[["type", "question", "gold", "pred", "gen_fill", "gen_match", "gen_sim"]]

,type,question,gold,pred,gen_fill,gen_match,gen_sim
0,network_protocol_requirements,네트워크 프로토콜 지원 요구는?,"['IPv4', 'IPv6']","CONTEXT에 명시된 네트워크 프로토콜(예: FTP, Telnet, Finger ...",1.0,0.0,4.379562
1,requirements_must,필수 요구사항(기능/성능/보안 등)은 무엇인가?,"{'SFR': {'title': '기능 요구사항', 'items': [{'id': ...","목표시스템이 반드시 수행하여야 하는 기능(총 38개 기능), 성능 요구사항(처리속도...",1.0,0.0,13.257215
2,contract_type,계약 방식(일반경쟁/제한경쟁/협상에 의한 계약 등)은 무엇인가?,제한경쟁입찰,협상에 의한 계약,1.0,0.0,13.333333
3,eval_items,평가 항목(기술/가격 등) 구성은 어떻게 되는가?,"{'overall': {'total': 100, 'composition': {'te...","기술평가(정성평가, 총 100점을 90점 만점으로 환산) 및 가격평가로 구성됨. 기...",1.0,0.0,20.224719
4,web_standard_compliance_items,웹표준(브라우저 관련) 준수 항목 예시는?,"['HTML 4.01', 'HTML 5', 'CSS 2.1', 'XHTML 1.0'...",전자정부 웹 표준 준수지침(행정안전부 고시) 준수; 전자정부서비스 호환성 준수지침(...,1.0,0.0,20.330969
5,pre_deadline_required_certificates,제출 마감일 전일까지 요구되는 확인서는?,중‧소기업 또는 소상공인 확인서,"벤처확인서(벤처확인 발급번호, 유효기간)",1.0,0.0,20.512821
6,subcontracting_requirements,하도급을 포함하면 반드시 지켜야 하는 것은?,"['하도급 사전승인', '하도급 비율 제한', '재하도급 불허']","하도급을 포함한 경우에는 하도급 사전승인(발주기관으로부터 사전승인), 하도급 비율 ...",1.0,0.0,21.935484
7,post_contract_submission_docs,계약체결 후 10일 이내 제출해야 하는 서류는?,"['착수계', '책임기술자 선임계', '책임기술자 이력서 및 기술자격증 사본', '...","대표자용 보안확약서, 참여자용 보안확약서, 보안서약서, 보안각서",1.0,0.0,25.000000
8,budget,총 사업 예산(사업비)은 얼마인가?,"352,000,000원",20억원미만,1.0,0.0,33.333333
9,price_eval,가격 평가 방식(최저가/협상 등)은 무엇인가?,협상에 의한 계약,가격평가는 「협상에 의한 계약체결 기준」제656호(기재부 계약예규)에 따름; 평가비...,1.0,0.0,36.363636


### save single JSON

In [8]:
outdir = Path(BASE_DIR) / "outputs_single_doc"
outdir.mkdir(parents=True, exist_ok=True)

safe_doc_id = re.sub(r"[^\w\-.]+", "_", str(doc_id))
outpath = outdir / f"exp{spec.exp_id:02d}_docidx{TARGET_DOC_INDEX:02d}_{safe_doc_id}.json"

payload: Dict[str, Any] = {
    "meta": {
        "target_doc_index": int(TARGET_DOC_INDEX),
        "target_doc_id": str(doc_id),
        "target_doc_path": str(target_doc_path),
        "spec": {
            "exp_id": int(spec.exp_id),
            "chunker": spec.chunker,
            "retriever": spec.retriever,
            "generator": spec.generator,
        },
        "top_k": int(top_k),
        "sim_threshold": int(SIM_THRESHOLD),
        "perquestionretrieve": bool(CONFIG.get("perquestionretrieve", True)),
        "perquestiongenerate": bool(CONFIG.get("perquestiongenerate", True)),
        "savecontextperquestion": bool(CONFIG.get("savecontextperquestion", False)),
        "common_doc_mark": COMMON_DOC_MARK,
    },
    "doc_metrics": doc_metrics,
    "per_question": detail_view.to_dict(orient="records"),
}

with open(outpath, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", outpath)

Saved: d:\dev\github\codeit-part3-team4\outputs_single_doc\exp03_docidx00__사_벤처기업협회_2024년_벤처확인종합관리시스템_기능_고도화_용역사업_.pdf.json


### optional: only bad cases

In [9]:
tmp = detail_view.copy()
tmp["pred_l"] = tmp["pred"].fillna("").astype(str).str.strip().str.lower()

bad_cases = tmp.loc[
    (tmp["gen_match"] == 0.0)
    | (tmp["pred_l"].isin(SENT_NOTFOUND_SET | SENT_GENFAIL_SET))
].reset_index(drop=True)

print("Bad cases:", len(bad_cases))
bad_cases[["type", "question", "gold", "pred", "gen_fill", "gen_match", "gen_sim"]].head(100)

Bad cases: 17


,type,question,gold,pred,gen_fill,gen_match,gen_sim
0,network_protocol_requirements,네트워크 프로토콜 지원 요구는?,"['IPv4', 'IPv6']","CONTEXT에 명시된 네트워크 프로토콜(예: FTP, Telnet, Finger ...",1.0,0.0,4.379562
1,requirements_must,필수 요구사항(기능/성능/보안 등)은 무엇인가?,"{'SFR': {'title': '기능 요구사항', 'items': [{'id': ...","목표시스템이 반드시 수행하여야 하는 기능(총 38개 기능), 성능 요구사항(처리속도...",1.0,0.0,13.257215
2,contract_type,계약 방식(일반경쟁/제한경쟁/협상에 의한 계약 등)은 무엇인가?,제한경쟁입찰,협상에 의한 계약,1.0,0.0,13.333333
3,eval_items,평가 항목(기술/가격 등) 구성은 어떻게 되는가?,"{'overall': {'total': 100, 'composition': {'te...","기술평가(정성평가, 총 100점을 90점 만점으로 환산) 및 가격평가로 구성됨. 기...",1.0,0.0,20.224719
4,web_standard_compliance_items,웹표준(브라우저 관련) 준수 항목 예시는?,"['HTML 4.01', 'HTML 5', 'CSS 2.1', 'XHTML 1.0'...",전자정부 웹 표준 준수지침(행정안전부 고시) 준수; 전자정부서비스 호환성 준수지침(...,1.0,0.0,20.330969
5,pre_deadline_required_certificates,제출 마감일 전일까지 요구되는 확인서는?,중‧소기업 또는 소상공인 확인서,"벤처확인서(벤처확인 발급번호, 유효기간)",1.0,0.0,20.512821
6,subcontracting_requirements,하도급을 포함하면 반드시 지켜야 하는 것은?,"['하도급 사전승인', '하도급 비율 제한', '재하도급 불허']","하도급을 포함한 경우에는 하도급 사전승인(발주기관으로부터 사전승인), 하도급 비율 ...",1.0,0.0,21.935484
7,post_contract_submission_docs,계약체결 후 10일 이내 제출해야 하는 서류는?,"['착수계', '책임기술자 선임계', '책임기술자 이력서 및 기술자격증 사본', '...","대표자용 보안확약서, 참여자용 보안확약서, 보안서약서, 보안각서",1.0,0.0,25.000000
8,budget,총 사업 예산(사업비)은 얼마인가?,"352,000,000원",20억원미만,1.0,0.0,33.333333
9,price_eval,가격 평가 방식(최저가/협상 등)은 무엇인가?,협상에 의한 계약,가격평가는 「협상에 의한 계약체결 기준」제656호(기재부 계약예규)에 따름; 평가비...,1.0,0.0,36.363636
